<a href="https://colab.research.google.com/github/mat1506/Datos/blob/main/python-basics/google-colab-r.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
%load_ext rpy2.ipython

The rpy2.ipython extension is already loaded. To reload it, use:
  %reload_ext rpy2.ipython


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!apt-get update -qq
!apt-get install -y -qq r-cran-remotes r-cran-sf r-cran-dplyr r-cran-tidyr r-cran-zoo libcurl4-openssl-dev libssl-dev libxml2-dev

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%R

cat("Forzando la descarga de binarios ultra-rápidos...\n")
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))

cat("Instalando remotes y paquetes de R...\n")
install.packages(c("remotes", "sf", "dplyr", "tidyr","zoo"))

cat("\n¡LIBRERÍAS LISTAS! Pasa a la siguiente celda.\n")

In [7]:
%%R
install.packages(c("rnaturalearth", "here", "carbondate"), repos="https://cloud.r-project.org")
remotes::install_github("xronos-ch/xronos.R", quiet = TRUE)
cat("\n¡AHORA SÍ, ENTORNO COMPLETADO!\n")


¡AHORA SÍ, ENTORNO COMPLETADO!


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/rnaturalearth_1.2.0.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/here_1.0.2.tar.gz'
trying URL 'https://cloud.r-project.org/src/contrib/carbondate_1.1.0.tar.gz'

The downloaded source packages are in
	‘/tmp/RtmpvvgWhE/downloaded_packages’


In [9]:
%%R
install.packages("rnaturalearthdata", repos="https://cloud.r-project.org")
# ==============================================================================
# SUPPLEMENTARY SCRIPT: PALEO-DEMOGRAPHIC RECONSTRUCTION
# Authors: Matías Frugone-Álvarez
# Target Journal: Nature Communications
# Description: Bayesian modeling of radiocarbon datasets to infer population
# dynamics across nested spatial scales using Variable-Rate Poisson Processes.
# ==============================================================================

# 1. INITIALIZATION & REPRODUCIBILITY
# ------------------------------------------------------------------------------
set.seed(42)

library(xronos)
library(rnaturalearth)
library(sf)
library(dplyr)
library(here)
library(tidyr)
library(carbondate)
library(zoo)

# Define Coordinate Reference Systems (CRS)
crs_wgs84 <- 4326
crs_utm19s <- 32719 # For northern Chile and Argentina only
crs_albers_sa <- "+proj=aea +lat_1=-5 +lat_2=-42 +lat_0=-32 +lon_0=-60 +x_0=0 +y_0=0 +datum=WGS84 +units=m +no_defs"

dir.create(here("analysis", "data", "raw_data"), recursive = TRUE, showWarnings = FALSE)
dir.create(here("analysis", "figures", "FigureS"), recursive = TRUE, showWarnings = FALSE)

# ==============================================================================
# 2. DATA ACQUISITION & STANDARDIZATION
# ==============================================================================
ptu <- read.csv(here("analysis", "data", "raw_data", "tulan.csv"))

sa_countries <- c("Argentina", "Bolivia", "Brazil", "Chile", "Colombia",
                  "Ecuador", "Paraguay", "Peru", "Uruguay", "Venezuela")

message("Downloading XRONOS data for South America...")
xronos_raw <- tryCatch({
  lapply(sa_countries, function(x) chron_data(country = x)) %>% bind_rows()
}, error = function(e) {
  stop("Error downloading XRONOS. Check connection or package installation.")
})

psa_clean <- xronos_raw %>%
  rename(C14Age = bp, C14SD = std, SiteID = site, LabID = labnr, Material = material) %>%
  mutate(
    C14Age = as.numeric(C14Age),
    C14SD  = as.numeric(C14SD),
    lat    = as.numeric(lat),
    lng    = as.numeric(lng),
    SiteID = gsub("_", "-", SiteID)
  ) %>%
  filter(!is.na(lat), !is.na(lng), !is.na(C14Age), !is.na(C14SD), C14Age > 0, C14SD > 0) %>%
  mutate(SiteID = ifelse(is.na(SiteID) | SiteID == "", paste0("Site-", row_number()), SiteID)) %>%
  distinct()

pts_sa_sf <- st_as_sf(psa_clean, coords = c("lng", "lat"), crs = crs_wgs84)
ptu_sf <- st_as_sf(ptu, coords = c("Lon","Lat"), crs = crs_wgs84)

# ==============================================================================
# 3. SPATIAL SUBSETTING & TAPHONOMIC FILTERING
# ==============================================================================
sam_poly     <- st_read(here("analysis", "data", "raw_data", "monsoon.shp"), quiet = TRUE) |> st_make_valid() |> st_transform(crs_wgs84)
atacama_poly <- st_read(here("analysis", "data", "raw_data", "AtacamaDesert2.shp"), quiet = TRUE) |> st_make_valid() |> st_transform(crs_wgs84)
coastal_poly <- st_read(here("analysis", "data", "raw_data", "costal.shp"), quiet = TRUE) |> st_make_valid() |> st_transform(crs_wgs84)
pdt_poly     <- st_read(here("analysis", "data", "raw_data", "PdT.shp"), quiet = TRUE) |> st_make_valid() |> st_transform(crs_wgs84)
rdl_poly     <- st_read(here("analysis", "data", "raw_data", "RdL.shp"), quiet = TRUE) |> st_make_valid() |> st_transform(crs_wgs84)
sda_poly     <- st_read(here("analysis", "data", "raw_data", "SdA.shp"), quiet = TRUE) |> st_make_valid() |> st_transform(crs_wgs84)
sa_poly      <- ne_countries(scale = "medium", returnclass = "sf", continent = "South America") %>% st_transform(crs_wgs84)

pts_sa_final <- st_filter(pts_sa_sf, sa_poly)
pts_sam      <- st_filter(pts_sa_sf, sam_poly)
pts_atacama  <- st_filter(pts_sa_sf, atacama_poly)
pts_coastal  <- st_filter(pts_sa_sf, coastal_poly)
pts_pdt      <- st_filter(pts_sa_sf, pdt_poly)
pts_rdl      <- st_filter(pts_sa_sf, rdl_poly)
pts_sda      <- st_filter(pts_sa_sf, sda_poly)

filter_terrestrial <- function(df) {
  if("Material" %in% names(df)) {
    marine_terms <- "shell|marine|whale|seal|fish|coral|seaweed|guano|mollusc"
    return(df %>% filter(!grepl(marine_terms, tolower(Material))))
  }
  return(df)
}

pts_sa_terr      <- filter_terrestrial(pts_sa_final)
pts_sam_terr     <- filter_terrestrial(pts_sam)
pts_atacama_terr <- filter_terrestrial(pts_atacama)
pts_coastal_terr <- filter_terrestrial(pts_coastal)
pts_pdt_terr     <- filter_terrestrial(pts_pdt)
pts_rdl_terr     <- filter_terrestrial(pts_rdl)
pts_sda_terr     <- filter_terrestrial(pts_sda)
pts_ptu_terr     <- filter_terrestrial(ptu_sf)

# Projections
pts_sa_proj  <- st_transform(pts_sa_terr, crs_albers_sa)
pts_sam_proj <- st_transform(pts_sam_terr, crs_albers_sa)
pts_ata_proj     <- st_transform(pts_atacama_terr, crs_utm19s)
pts_coastal_proj <- st_transform(pts_coastal_terr, crs_utm19s)
pts_pdt_proj     <- st_transform(pts_pdt_terr, crs_utm19s)
pts_rdl_proj     <- st_transform(pts_rdl_terr, crs_utm19s)
pts_sda_proj     <- st_transform(pts_sda_terr, crs_utm19s)
pts_ptu_proj     <- st_transform(pts_ptu_terr, crs_utm19s)

cat(sprintf("\nTerrestrial Datasets (N):\n SA: %d | SAM: %d | Atacama: %d | Coastal: %d | PdT: %d | RdL: %d | SdA: %d | Tulan: %d\n",
            nrow(pts_sa_proj), nrow(pts_sam_proj), nrow(pts_ata_proj), nrow(pts_coastal_proj), nrow(pts_pdt_proj), nrow(pts_rdl_proj), nrow(pts_sda_proj), nrow(pts_ptu_proj)))

# ==============================================================================
# 4. RAW DATA EXPORT (CSV & GeoPackage)
# ==============================================================================
cat("\nExporting spatial data to GeoPackage and CSV...\n")

flatten_for_export <- function(sf_object, is_csv = FALSE) {
  geom_col <- attr(sf_object, "sf_column")
  for (col in names(sf_object)) {
    if (col != geom_col && is.list(sf_object[[col]])) {
      sf_object[[col]] <- sapply(sf_object[[col]], toString)
    }
  }
  if (is_csv) return(st_drop_geometry(sf_object))
  return(sf_object)
}

# Export CSVs
write.csv(flatten_for_export(pts_sa_proj, TRUE),      here("analysis","data", "raw_data", "South_America_Total.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_sam_proj, TRUE),     here("analysis","data", "raw_data", "SAM_Monsoon.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_ata_proj, TRUE),     here("analysis","data", "raw_data", "AtacamaDesert2.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_coastal_proj, TRUE), here("analysis","data", "raw_data", "Coastal.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_pdt_proj, TRUE),     here("analysis","data", "raw_data", "PdT.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_rdl_proj, TRUE),     here("analysis","data", "raw_data", "RdL.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_sda_proj, TRUE),     here("analysis","data", "raw_data", "SdA.csv"), row.names = FALSE)
write.csv(flatten_for_export(pts_ptu_proj, TRUE),     here("analysis","data", "raw_data", "Tulan.csv"), row.names = FALSE)

# Export GeoPackages
st_write(flatten_for_export(pts_sa_proj),      here("analysis","data", "raw_data", "South_America_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_sam_proj),     here("analysis","data", "raw_data", "SAM_Monsoon_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_ata_proj),     here("analysis","data", "raw_data", "AtacamaDesert2_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_coastal_proj), here("analysis","data", "raw_data", "Coastal_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_pdt_proj),     here("analysis","data", "raw_data", "PdT_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_rdl_proj),     here("analysis","data", "raw_data", "RdL_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_sda_proj),     here("analysis","data", "raw_data", "SdA_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)
st_write(flatten_for_export(pts_ptu_proj),     here("analysis","data", "raw_data", "Tulan_Dates.gpkg"), delete_dsn = TRUE, quiet = TRUE)


Terrestrial Datasets (N):
 SA: 17380 | SAM: 2436 | Atacama: 1034 | Coastal: 1992 | PdT: 196 | RdL: 274 | SdA: 238 | Tulan: 61

Exporting spatial data to GeoPackage and CSV...


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)
trying URL 'https://cloud.r-project.org/src/contrib/rnaturalearthdata_1.0.0.tar.gz'
Content type 'application/x-gzip' length 3313555 bytes (3.2 MB)
downloaded 3.2 MB


The downloaded source packages are in
	‘/tmp/RtmpvvgWhE/downloaded_packages’


In [ ]:
%%R

# ==============================================================================
# 5. BAYESIAN NON-PARAMETRIC MODELING (carbondate)
# ==============================================================================
n_iter_mcmc <- 100000
n_thin_mcmc <- 50

# Ensure base directory exists in Drive
base_dir <- here("analysis", "data", "models")
dir.create(base_dir, recursive = TRUE, showWarnings = FALSE)

model_file <- file.path(base_dir, "bayesian_mcmc_pp_models.RData")

if (file.exists(model_file)) {

  cat("\n==================================================================\n")
  cat(" ARCHIVO ENCONTRADO: Cargando modelos bayesianos guardados...\n")
  cat("==================================================================\n")
  load(model_file)

} else {

  cat("\n==================================================================\n")
  cat(" ARCHIVO NO ENCONTRADO: Ejecutando modelos DPMM y Poisson...\n")
  cat(" ADVERTENCIA: Esto tomará varias horas en Google Colab. Por favor, espere.\n")
  cat("==================================================================\n")

  # --- 1. Dirichlet Process Mixture Models (DPMM) ---
  cat("\nIniciando fase 1: DPMM (Densidad Calendárica)...\n")
  mcmc_ptu     <- PolyaUrnBivarDirichlet(pts_ptu_proj$C14Age, pts_ptu_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_sda     <- PolyaUrnBivarDirichlet(pts_sda_proj$C14Age, pts_sda_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_rdl     <- PolyaUrnBivarDirichlet(pts_rdl_proj$C14Age, pts_rdl_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_pdt     <- PolyaUrnBivarDirichlet(pts_pdt_proj$C14Age, pts_pdt_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_coastal <- PolyaUrnBivarDirichlet(pts_coastal_proj$C14Age, pts_coastal_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_ata     <- PolyaUrnBivarDirichlet(pts_ata_proj$C14Age, pts_ata_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_sam     <- PolyaUrnBivarDirichlet(pts_sam_proj$C14Age, pts_sam_proj$C14SD, shcal20, n_iter = n_iter_mcmc, show_progress = TRUE)
  mcmc_sa      <- PolyaUrnBivarDirichlet(pts_sa_proj$C14Age, pts_sa_proj$C14SD, shcal20, n_iter = n_iter_mcmc, n_thin = n_thin_mcmc, show_progress = TRUE)

  # !!! CHECKPOINT: Guardar mitad del progreso !!!
  cat("\n[CHECKPOINT]: Guardando modelos DPMM en Drive antes de continuar...\n")
  save(mcmc_ptu, mcmc_sda, mcmc_rdl, mcmc_pdt, mcmc_coastal, mcmc_ata, mcmc_sam, mcmc_sa, file = model_file)

  # --- 2. Variable-Rate Poisson Process ---
  cat("\nIniciando fase 2: Poisson Process (Tasas de Ocurrencia)...\n")
  common_age_range <- c(0, 8500)

  pp_ptu     <- PPcalibrate(pts_ptu_proj$C14Age, pts_ptu_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_sda     <- PPcalibrate(pts_sda_proj$C14Age, pts_sda_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_rdl     <- PPcalibrate(pts_rdl_proj$C14Age, pts_rdl_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_pdt     <- PPcalibrate(pts_pdt_proj$C14Age, pts_pdt_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_coastal <- PPcalibrate(pts_coastal_proj$C14Age, pts_coastal_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_ata     <- PPcalibrate(pts_ata_proj$C14Age, pts_ata_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_sam     <- PPcalibrate(pts_sam_proj$C14Age, pts_sam_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, show_progress = TRUE)
  pp_sa      <- PPcalibrate(pts_sa_proj$C14Age, pts_sa_proj$C14SD, shcal20, calendar_age_range = common_age_range, n_iter = n_iter_mcmc, n_thin = n_thin_mcmc, show_progress = TRUE)

  # --- 3. GUARDAR RESULTADOS COMPLETOS ---
  cat("\n==================================================================\n")
  cat(" CÁLCULOS TERMINADOS: Guardando TODOS los modelos en Drive...\n")
  save(mcmc_ptu, mcmc_sda, mcmc_rdl, mcmc_pdt, mcmc_coastal, mcmc_ata, mcmc_sam, mcmc_sa,
       pp_ptu, pp_sda, pp_rdl, pp_pdt, pp_coastal, pp_ata, pp_sam, pp_sa,
       file = model_file)
  cat(" Modelos guardados exitosamente.\n")
  cat("==================================================================\n")
}

# ==============================================================================
# 6. DATA EXTRACTION & NORMALIZATION
# ==============================================================================
cal_ages <- seq(0, 7300, by = 1)

# Extract DPMM Densities
dens_ptu     <- FindPredictiveCalendarAgeDensity(mcmc_ptu, cal_ages)
dens_sda     <- FindPredictiveCalendarAgeDensity(mcmc_sda, cal_ages)
dens_rdl     <- FindPredictiveCalendarAgeDensity(mcmc_rdl, cal_ages)
dens_pdt     <- FindPredictiveCalendarAgeDensity(mcmc_pdt, cal_ages)
dens_coastal <- FindPredictiveCalendarAgeDensity(mcmc_coastal, cal_ages)
dens_ata     <- FindPredictiveCalendarAgeDensity(mcmc_ata, cal_ages)
dens_sam     <- FindPredictiveCalendarAgeDensity(mcmc_sam, cal_ages)
dens_sa      <- FindPredictiveCalendarAgeDensity(mcmc_sa, cal_ages)

# Extract Poisson Rates
rate_ptu     <- FindPosteriorMeanRate(pp_ptu, cal_ages)
rate_sda     <- FindPosteriorMeanRate(pp_sda, cal_ages)
rate_rdl     <- FindPosteriorMeanRate(pp_rdl, cal_ages)
rate_pdt     <- FindPosteriorMeanRate(pp_pdt, cal_ages)
rate_coastal <- FindPosteriorMeanRate(pp_coastal, cal_ages)
rate_ata     <- FindPosteriorMeanRate(pp_ata, cal_ages)
rate_sam     <- FindPosteriorMeanRate(pp_sam, cal_ages)
rate_sa      <- FindPosteriorMeanRate(pp_sa, cal_ages)

norm_data <- function(df, is_rate = FALSE) {
  val_col <- ifelse(is_rate, "rate_mean", "density_mean")
  low_col <- ifelse(is_rate, "rate_ci_lower", "density_ci_lower")
  up_col  <- ifelse(is_rate, "rate_ci_upper", "density_ci_upper")

  m <- max(df[[val_col]], na.rm = TRUE)
  data.frame(
    mean  = df[[val_col]] / m,
    lower = df[[low_col]] / m,
    upper = df[[up_col]] / m
  )
}

nd_ptu     <- norm_data(dens_ptu)
nd_sa      <- norm_data(dens_sa);      nr_sa      <- norm_data(rate_sa, TRUE)
nd_sam     <- norm_data(dens_sam);     nr_sam     <- norm_data(rate_sam, TRUE)
nd_ata     <- norm_data(dens_ata);     nr_ata     <- norm_data(rate_ata, TRUE)
nd_rdl     <- norm_data(dens_rdl);     nr_rdl     <- norm_data(rate_rdl, TRUE)
nd_pdt     <- norm_data(dens_pdt);     nr_pdt     <- norm_data(rate_pdt, TRUE)
nd_coastal <- norm_data(dens_coastal); nr_coastal <- norm_data(rate_coastal, TRUE)
nd_sda     <- norm_data(dens_sda);     nr_sda     <- norm_data(rate_sda, TRUE)

# ==============================================================================
# 7. DEMOGRAPHIC GROWTH RATE (r) USING DPMM DENSITIES
# ==============================================================================
StableBayesianGrowth <- function(dens_df, step = 50, smooth_window = 150, mask_threshold = 0.005) {

  dens_df <- dens_df %>% filter(calendar_age_BP >= 200 & calendar_age_BP <= 8300)

  max_dens <- max(dens_df$density_mean, na.rm = TRUE)
  eps <- max_dens * 1e-6
  threshold <- max_dens * mask_threshold

  breaks <- seq(max(dens_df$calendar_age_BP), min(dens_df$calendar_age_BP), by = -step)

  S_mean_raw <- approx(dens_df$calendar_age_BP, dens_df$density_mean, xout = breaks)$y

  ma_window <- max(2, round(smooth_window / step))
  S_mean <- zoo::rollmean(S_mean_raw, k = ma_window, fill = NA, align = "center")

  S_low <- approx(dens_df$calendar_age_BP, dens_df$density_ci_lower, xout = breaks)$y
  S_up  <- approx(dens_df$calendar_age_BP, dens_df$density_ci_upper, xout = breaks)$y
  rel_err <- pmax(0, (S_up - S_low) / (2 * (S_mean_raw + eps)), na.rm = TRUE)

  r_m <- log((S_mean[-1] + eps) / (S_mean[-length(S_mean)] + eps)) / step
  r_err <- (rel_err[-1] / step) * 1.5

  r_l <- r_m - r_err
  r_u <- r_m + r_err

  valid_mask <- S_mean[-1] >= threshold & !is.na(r_m)
  r_m[!valid_mask] <- NA
  r_l[!valid_mask] <- NA
  r_u[!valid_mask] <- NA

  data.frame(
    t_mid = (breaks[-length(breaks)] + breaks[-1]) / 2,
    mean  = r_m,
    lower = r_l,
    upper = r_u
  )
}

r_ptu     <- StableBayesianGrowth(dens_ptu, step = 50, smooth_window = 150)
r_sa      <- StableBayesianGrowth(dens_sa, step = 50, smooth_window = 150)
r_sam     <- StableBayesianGrowth(dens_sam, step = 50, smooth_window = 150)
r_ata     <- StableBayesianGrowth(dens_ata, step = 50, smooth_window = 150)
r_rdl     <- StableBayesianGrowth(dens_rdl, step = 50, smooth_window = 150)
r_pdt     <- StableBayesianGrowth(dens_pdt, step = 50, smooth_window = 150)
r_coastal <- StableBayesianGrowth(dens_coastal, step = 50, smooth_window = 150)
r_sda     <- StableBayesianGrowth(dens_sda, step = 50, smooth_window = 150)




 ARCHIVO NO ENCONTRADO: Ejecutando modelos DPMM y Poisson...
 ADVERTENCIA: Esto tomará varias horas en Google Colab. Por favor, espere.

Iniciando fase 1: DPMM (Densidad Calendárica)...
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%

[CHECKPOINT]: Guardando modelos DPMM en Drive antes de continuar...

Iniciando fase 2: Poisson Process (Tasas de Ocurrencia)...
  |=========================================

In [ ]:
# ==============================================================================
# 8. MULTIPANEL VISUALIZATIONS & HIGH-RES EXPORT
# ==============================================================================
plot_band <- function(x, lower, upper, color, alpha = 0.3) {
  valid <- !is.na(lower) & !is.na(upper)
  if(any(valid)) {
    runs <- rle(valid)
    ends <- cumsum(runs$lengths)
    starts <- c(1, ends[-length(ends)] + 1)

    for(i in seq_along(runs$values)) {
      if(runs$values[i]) {
        idx <- starts[i]:ends[i]
        if(length(idx) > 1) {
          polygon(c(x[idx], rev(x[idx])), c(lower[idx], rev(upper[idx])),
                  col = adjustcolor(color, alpha.f = alpha), border = NA)
        }
      }
    }
  }
}

get_sym_ylim <- function(data) {
  valid_data <- subset(data, t_mid >= 400 & t_mid <= 7000)
  vals <- c(valid_data$lower, valid_data$upper)
  vals <- vals[!is.na(vals) & !is.infinite(vals)]

  if(length(vals) == 0) return(c(-0.005, 0.005))

  max_val <- max(abs(vals), na.rm = TRUE) * 1.2
  max_val <- max(max_val, 0.003)
  max_val <- min(max_val, 0.015)
  return(c(-max_val, max_val))
}

plot_metric_panel <- function(x_data, y_data, color, title, xlim_val, ylim_val, ylab_text, show_x_axis = FALSE, is_r = FALSE) {
  plot(x_data, y_data$mean, type="n", xlim=xlim_val, ylim=ylim_val, xlab="", ylab=ylab_text, xaxt="n", cex.axis=1.2, cex.lab=1.3)

  x_ticks <- seq(7500, 0, by = -500)
  abline(v = x_ticks, col = adjustcolor("lightgray", alpha.f=0.6), lty = "dotted")
  grid(nx = NA, ny = NULL)

  if(is_r) {
    polygon(c(8000, -100, -100, 8000), c(0.0001, 0.0001, -0.0001, -0.0001), col=adjustcolor("black", alpha.f=0.1), border=NA)
    abline(h=0, lty=2, lwd=1.5)
  } else {
    abline(h=0, lty=1, col="gray")
  }

  plot_band(x_data, y_data$lower, y_data$upper, color)
  lines(x_data, y_data$mean, lwd=3, col=color)
  legend("topright", legend=title, text.col=color, bty="n", text.font=2, cex=1.3)

  if (show_x_axis) axis(1, at = x_ticks, labels = x_ticks, cex.axis = 1.2)
}

cat("\nGenerating and exporting publication-quality figures to /analysis/figures/FigureS/ ...\n")

# --- PLOT A: Predictive Calendar Density (DPMM) ---
pdf(here("analysis", "figures", "FigureS", "FigureS_A_PredictiveDensity.pdf"), width = 10, height = 16)
par(mfrow = c(8, 1), mar = c(0.5, 5, 0.5, 1), oma = c(4, 1, 3, 1))
common_ylim_dens <- c(0, max(c(nd_sa$upper, nd_sam$upper, nd_ata$upper, nd_coastal$upper, nd_pdt$upper, nd_rdl$upper, nd_sda$upper, nd_ptu$upper), na.rm=TRUE) * 1.05)

plot_metric_panel(cal_ages, nd_sa,      "darkgrey",   "South America Total (DPMM)", c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_sam,     "steelblue",  "SAM Monsoon (DPMM)",         c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_ata,     "darkorange", "Atacama Desert (DPMM)",      c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_coastal, "purple",     "Coastal Region (DPMM)",      c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_pdt,     "darkgreen",  "Pampa del Tamarugal (DPMM)", c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_rdl,     "brown",      "Rio Loa Basin (DPMM)",       c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_sda,     "magenta",    "Salar de Atacama (DPMM)",    c(7300, 0), common_ylim_dens, "Density")
plot_metric_panel(cal_ages, nd_ptu,     "red",        "Tulan (DPMM)",               c(7300, 0), common_ylim_dens, "Density", show_x_axis = TRUE)
mtext("Predictive Calendar Density (95% Credible Intervals)", side = 3, outer = TRUE, cex = 1.5, font = 2)
mtext("cal BP Years", side = 1, outer = TRUE, cex = 1.2, line = 2.5)
dev.off()

# --- PLOT B: Activity Occurrence Rates (Poisson) ---
pdf(here("analysis", "figures", "FigureS", "FigureS_B_ActivityRates.pdf"), width = 10, height = 16)
par(mfrow = c(8, 1), mar = c(0.5, 5, 0.5, 1), oma = c(4, 1, 3, 1))
common_ylim_rate <- c(0, max(c(nr_sa$upper, nr_sam$upper, nr_ata$upper, nr_coastal$upper, nr_pdt$upper, nr_rdl$upper, nr_sda$upper, nr_ptu$upper), na.rm=TRUE) * 1.05)

plot_metric_panel(cal_ages, nr_sa,      "darkgrey",   "South America Total (Poisson)", c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_sam,     "steelblue",  "SAM Monsoon (Poisson)",         c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_ata,     "darkorange", "Atacama Desert (Poisson)",      c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_coastal, "purple",     "Coastal Region (Poisson)",      c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_pdt,     "darkgreen",  "Pampa del Tamarugal (Poisson)", c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_rdl,     "brown",      "Rio Loa Basin (Poisson)",       c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_sda,     "magenta",    "Salar de Atacama (Poisson)",    c(7300, 0), common_ylim_rate, "Rate")
plot_metric_panel(cal_ages, nr_ptu,     "red",        "Tulan (Poisson)",               c(7300, 0), common_ylim_rate, "Rate", show_x_axis = TRUE)
mtext("Activity Occurrence Rates (95% Credible Intervals)", side = 3, outer = TRUE, cex = 1.5, font = 2)
mtext("cal BP Years", side = 1, outer = TRUE, cex = 1.2, line = 2.5)
dev.off()

# --- PLOT C: Per Capita Growth Rate (r) ---
pdf(here("analysis", "figures", "FigureS", "FigureS_C_GrowthRates.pdf"), width = 10, height = 16)
par(mfrow = c(8, 1), mar = c(0.5, 5.5, 0.5, 1), oma = c(4, 1, 3, 1))

plot_metric_panel(r_sa$t_mid,      r_sa,      "darkgrey",   "South America Total", c(7300, 200), get_sym_ylim(r_sa),      "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_sam$t_mid,     r_sam,     "steelblue",  "SAM Monsoon",         c(7300, 200), get_sym_ylim(r_sam),     "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_ata$t_mid,     r_ata,     "darkorange", "Atacama Desert",      c(7300, 200), get_sym_ylim(r_ata),     "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_coastal$t_mid, r_coastal, "purple",     "Coastal Region",      c(7300, 200), get_sym_ylim(r_coastal), "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_pdt$t_mid,     r_pdt,     "darkgreen",  "Pampa del Tamarugal", c(7300, 200), get_sym_ylim(r_pdt),     "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_rdl$t_mid,     r_rdl,     "brown",      "Rio Loa Basin",       c(7300, 200), get_sym_ylim(r_rdl),     "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_sda$t_mid,     r_sda,     "magenta",    "Salar de Atacama",    c(7300, 200), get_sym_ylim(r_sda),     "r (yr⁻¹)", is_r = TRUE)
plot_metric_panel(r_ptu$t_mid,     r_ptu,     "red",        "Tulan",               c(7300, 200), get_sym_ylim(r_ptu),     "r (yr⁻¹)", show_x_axis = TRUE, is_r = TRUE)
mtext("Demographic Growth Rates (95% Credible Intervals)", side = 3, outer = TRUE, cex = 1.5, font = 2)
mtext("cal BP Years", side = 1, outer = TRUE, cex = 1.2, line = 2.5)
dev.off()

cat("\nAnalysis complete. All figures and spatial data exported successfully.\n")